## Parquet File Read

In [0]:
from pyspark.sql.functions import col, count, sum, when

df = spark.read.parquet("data/result_ecommerce_sales.parquet")

df.display()

order_id,customer_id,product_name,category,region,quantity,unit_price,revenue,payment_method,order_date,revenue_category
ORD10000,CUST1031,LAPTOP,Electronics,South,1,613,613,UPI,2024-11-23,Medium
ORD10001,CUST1054,CAMERA,Electronics,North,1,1259,1259,Credit Card,2024-02-22,High
ORD10002,CUST1077,MOBILE,Electronics,North,2,1084,2168,UPI,2024-02-17,High
ORD10003,CUST1057,CAMERA,Electronics,East,4,501,2004,Credit Card,2024-11-28,High
ORD10004,CUST1019,HEADPHONES,Accessories,South,3,619,1857,Debit Card,2024-03-22,High
ORD10005,CUST1045,LAPTOP,Electronics,East,4,248,992,Debit Card,2024-02-22,Medium
ORD10006,CUST1118,HEADPHONES,Accessories,West,5,305,1525,Credit Card,2024-01-23,High
ORD10007,CUST1073,TABLET,Electronics,South,5,790,3950,Credit Card,2024-10-09,High
ORD10008,CUST1109,MOBILE,Electronics,South,3,213,639,Credit Card,2024-01-24,Medium
ORD10009,CUST1106,TABLET,Electronics,East,4,1351,5404,UPI,2024-07-13,High


## Duplicate order_id Check

In [0]:
duplicates = df.groupBy("order_id") \
               .count() \
               .filter(col("count") > 1)

duplicates.display()

order_id,count


In [0]:
duplicate_count = duplicates.count()
print("Duplicate order_id records:", duplicate_count)

Duplicate order_id records: 0


## Missing Values Check

In [0]:
missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.display()

order_id,customer_id,product_name,category,region,quantity,unit_price,revenue,payment_method,order_date,revenue_category
0,0,0,0,0,0,0,0,0,0,0


## Validate quantity > 0

In [0]:
invalid_quantity = df.filter(col("quantity") <= 0)

invalid_quantity.display()

print("Invalid quantity records:", invalid_quantity.count())

order_id,customer_id,product_name,category,region,quantity,unit_price,revenue,payment_method,order_date,revenue_category


Invalid quantity records: 0


## Validate revenue = quantity × unit_price

In [0]:
invalid_revenue = df.filter(
    col("revenue") != col("quantity") * col("unit_price")
)

invalid_revenue.display()

print("Invalid revenue records:", invalid_revenue.count())

order_id,customer_id,product_name,category,region,quantity,unit_price,revenue,payment_method,order_date,revenue_category


Invalid revenue records: 0


Data Quality Summary Report

In [0]:
total_rows = df.count()

duplicate_rows = df.groupBy("order_id") \
                   .count() \
                   .filter(col("count") > 1) \
                   .count()

invalid_qty = df.filter(col("quantity") <= 0).count()

invalid_rev = df.filter(
    col("revenue") != col("quantity") * col("unit_price")
).count()

print("------ DATA QUALITY REPORT ------")
print("Total Rows:", total_rows)
print("Duplicate order_id:", duplicate_rows)
print("Invalid Quantity:", invalid_qty)
print("Invalid Revenue:", invalid_rev)

------ DATA QUALITY REPORT ------
Total Rows: 500
Duplicate order_id: 0
Invalid Quantity: 0
Invalid Revenue: 0
